# Homework 5 - API Practice

## PART 3: Exploratory Data Analysis

### a. Join Last.fm and Genius

In [1]:
import pandas as pd

# load two datasets
lastfm = pd.read_csv("../data/lastfm.csv")
genius = pd.read_csv("../data/genius.csv")

This block combines the datasets using album and song_name. A left join retains every Last.fm track and adds the corresponding Genius information. The `validate="one_to_one"` argument checks that each combination of album and song name is unique in both datasets, preventing duplicate matches from multiplying rows.

In [2]:
# merge the lastfm and genius datasets on album and song_name
combined = lastfm.merge(genius, on=["album", "song_name"], how="left", validate="one_to_one")
combined.head()

,album,song_name,track_number,duration_seconds,track_playcount,track_listeners,lyrics,writer_artists,producer_artists
0,11月的蕭邦,夜曲,1,229,172050,16550,一群嗜血的螞蟻 被腐肉所吸引\n我面無表情 看孤獨的風景\n失去妳 愛恨開始分明\n失去妳 ...,方文山 (Vincent Fang),周杰倫 (Jay Chou)
1,11月的蕭邦,藍色風暴,2,284,27143,4345,亙古長夜 善惡交接 我終於懂得流眼淚\n洪荒世界 百分之七十都是水 湛藍的曠野\n在海平面渴...,方文山 (Vincent Fang),周杰倫 (Jay Chou)
2,11月的蕭邦,髮如雪,3,299,101027,10018,狼牙月 伊人憔悴\n我舉杯 飲盡了風雪\n是誰打翻前世櫃 惹塵埃是非\n緣字訣 幾番輪迴\n...,方文山 (Vincent Fang),周杰倫 (Jay Chou)
3,11月的蕭邦,黑色毛衣,4,240,85914,10311,一件黑色毛衣 兩個人的回憶\n雨過之後 更難忘記 忘記我還愛你\n你不用在意 流淚也只是剛好...,周杰倫 (Jay Chou),周杰倫 (Jay Chou)
4,11月的蕭邦,四面楚歌,5,244,49198,6837,我的生活像 拍了一齣戲\n有超多導演跟編劇 只說了台詞一句\n而他們配了八百個語氣 操控著我...,周杰倫 (Jay Chou),周杰倫 (Jay Chou)


In [3]:
# save the combined dataset to a csv file
combined.to_csv("../data/combined.csv", index=False)

### b. Insight 1: Audience size and average listening frequency reveal different aspects of a song’s popularity

The following blocks create a copy of the combined data and calculate plays per listener by dividing each song’s play count by its listener count. Listener count describes the size of the recorded audience, while plays per listener describes the average number of recorded plays associated with that audience.

In [4]:
# create a copy of the combined dataframe for analysis
analysis = combined.copy()

In [5]:
# calculate plays per listener for each track
analysis["plays_per_listener"] = (analysis["track_playcount"] / analysis["track_listeners"])

# set the columns to display for analysis
columns = ["song_name", "album", "track_playcount", "track_listeners", "plays_per_listener"]

In [6]:
# find the top 5 tracks with the highest audience
top_audience = (analysis.nlargest(5, "track_listeners")[columns].round(2))
top_audience

,song_name,album,track_playcount,track_listeners,plays_per_listener
11,一路向北,11月的蕭邦,145875,24839,5.87
34,晴天,葉惠美,201040,18120,11.09
13,七里香,七里香,179221,17690,10.13
0,夜曲,11月的蕭邦,172050,16550,10.40
24,青花瓷,我很忙,158622,15647,10.14


In [7]:
# find the top 5 tracks with the highest plays per listener
top_plays_per_listener = (analysis.nlargest(5, "plays_per_listener")[columns].round(2))
top_plays_per_listener

,song_name,album,track_playcount,track_listeners,plays_per_listener
28,我不配,我很忙,129860,10766,12.06
48,說好的幸福呢,魔杰座,120592,10565,11.41
53,稻香,魔杰座,149984,13386,11.20
34,晴天,葉惠美,201040,18120,11.09
23,彩虹,我很忙,133325,12213,10.92


In [8]:
# calculate the median plays per listener
print("Median plays per listener:", round(analysis["plays_per_listener"].median(), 2))

Median plays per listener: 7.99


“一路向北” has the largest recorded audience in the dataset, with 24,839 listeners and 145,875 plays. However, its average of 5.87 plays per listener is below the dataset median of approximately 7.99. This shows that reaching a large audience does not necessarily produce a high average number of plays per listener.

“我不配” has the highest average plays per listener at approximately 12.06, followed by “說好的幸福呢” at 11.41 and “稻香” at 11.20. “我不配” has 10,766 listeners, a smaller audience than “一路向北,” but a substantially higher average listening frequency.

These results show why both measures are useful. Listener count captures audience reach, while plays per listener adds information about the volume of listening relative to that audience. We need to note that the ratio here is an aggregate average and may be influenced by heavy listeners.

### c. Insight 2: Concentrated writer representation and producer

In [9]:
# print number of songs and albums
print("Number of songs:", len(combined))
print("Number of albums:", combined["album"].nunique())

# check for missing values 
combined.isna().sum()

Number of songs: 54
Number of albums: 5


album               0
song_name           0
track_number        0
duration_seconds    0
track_playcount     0
track_listeners     0
lyrics              0
writer_artists      2
producer_artists    0
dtype: int64

In [10]:
# summarize writer-level statistics
writer = (
    combined["writer_artists"]
        .dropna()
        .str.split(";")
        .explode()
        .str.strip()
)

# count the number of songs with a writer specified
songs_writer = combined["writer_artists"].notna().sum()

# create a summary of the number of songs per writer
writer_summary = (
    writer.value_counts()
          .rename_axis("writer")
          .reset_index(name="number_of_songs")
)

# calculate the percentage of songs with each writer specified
writer_summary["percentage_of_songs_writer"] = (writer_summary["number_of_songs"] / songs_writer * 100).round(2)
writer_summary

,writer,number_of_songs,percentage_of_songs_writer
0,方文山 (Vincent Fang),31,59.62
1,周杰倫 (Jay Chou),13,25.00
2,黃俊郎 (Alang Huang),6,11.54
3,Devon Song,1,1.92
4,宋健彰 (Song Jian Zhang),1,1.92
5,曾郁婷 (Sissel Tseng),1,1.92
6,徐若瑄 (Vivian Hsu),1,1.92


In [11]:
# summarize producer-level statistics
producer_credits = (
    combined["producer_artists"]
        .dropna()
        .str.split(";")
        .explode()
        .str.strip()
)

# count the number of songs with a producer specified
producer_summary = (
    producer_credits.value_counts()
                    .rename_axis("producer")
                    .reset_index(name="number_of_songs")
)

producer_summary

,producer,number_of_songs
0,周杰倫 (Jay Chou),54


Genius provides writer credits for 52 of the 54 songs. Vincent Fang appears most frequently, with credits on 31 songs, representing approximately 59.62% of the songs with available writer information. Jay Chou appears on 13 songs, or 25%, followed by Alang Huang on 6 songs, or 11.54%. This indicates that a small group of recurring contributors accounts for much of the recorded writer representation, which means that Jay Chou and Vincent Fang have a close cooperative relationship with each other.

Producer credits are more concentrated. All 54 songs with available producer information list Jay Chou as producer. Taken together, the two tables suggest a recurring group of credited writers alongside consistent producer representation by Jay Chou.